# الدرس 02: بناء محرك التقطيع الدلالي المتقدم بلغة بايثون
### Practical Semantic Chunking Engine in Pure Python (Lecture 38)

مرحبًا بك في الدرس الثاني من مرحلة **تقنيات التقطيع الدلالي المتقدم (`Advanced Semantic Chunking`)**.

في هذا الدرس العملي، نقوم ببناء خوارزمية التقطيع الدلالي بالكامل من الصفر بلغة بايثون دون الاعتماد على سلاسل مغلقة (`Black-box Chains`)، باستخدام:
1. `sentence-transformers` وموديل `all-MiniLM-L6-v2` لتوليد متجهات الجمل.
2. `scikit-learn` لحساب مصفوفة التشابه الزوجي (`Cosine Similarity`).
3. خوارزمية التجميع الديناميكي وكشف نقاط الانفصال (`Breakpoints`) بناءً على العتبة (`Threshold`).

---

In [1]:
# 1. استدعاء المكتبات الأساسية
import os
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ تم استدعاء المكتبات بنجاح!")

✅ تم استدعاء المكتبات بنجاح!


---
## الخطوة 1: تحميل موديل التضمين المحلي (`Local Sentence Embedding Model`)

نستخدم موديل `all-MiniLM-L6-v2` من عائلة `SentenceTransformers`:
- خفيف وسريع جداً محلياً (384 أبعاد).
- مثالي لعمليات التقطيع الدلالي في الـ Production دون تكلفة APIs خارجية.

In [2]:
# تحميل الموديل محلياً
model = SentenceTransformer("all-MiniLM-L6-v2")

print("✅ تم تحميل موديل SentenceTransformer بنجاح!")
print(f"أبعاد المتجه (Embedding Dimension): {model.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ تم تحميل موديل SentenceTransformer بنجاح!
أبعاد المتجه (Embedding Dimension): 384


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_3276\1414473632.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"أبعاد المتجه (Embedding Dimension): {model.get_sentence_embedding_dimension()}")


---
## الخطوة 2: قراءة البيانات الخام وتجزئة الجمل (`Document Segmentation`)

نقوم بقراءة النص من الملف النصي `sample_text.txt` وتقسيم النص إلى جمل مستقلة مع إزالة المسافات الفارغة الزائدة (`strip`).

In [3]:
# قراءة النص من الملف sample_text.txt
with open("sample_text.txt", "r", encoding="utf-8") as f:
    text = f.read()

# تقسيم النص إلى أسطر/جمل مستقلة
sentences = [s.strip() for s in text.split("\n") if s.strip()]

print(f"📊 إجمالي عدد الجمل المستخرجة: {len(sentences)}\n")
for idx, s in enumerate(sentences, 1):
    print(f"[{idx}] {s}")

📊 إجمالي عدد الجمل المستخرجة: 5

[1] LangChain is a framework for building applications with LLMs.
[2] LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.
[3] You can create chains, agents, memory, and retrievers.
[4] The Eiffel Tower is located in Paris.
[5] France is a popular tourist destination.


---
## الخطوة 3: توليد التضمينات لكل جملة (`Sentence Embeddings`)

نقوم بتمرير قائمة الجمل إلى الموديل دفعة واحدة عبر `model.encode(sentences)`.

In [4]:
# حساب المتجهات الكثيفة
embeddings = model.encode(sentences)

print(f"أبعاد مصفوفة التضمين: {embeddings.shape} -> ({len(sentences)} جمل × 384 بُعداً)")

أبعاد مصفوفة التضمين: (5, 384) -> (5 جمل × 384 بُعداً)


---
## الخطوة 4: قياس التشابه الزوجي بين الجمل المتجاورة (`Pairwise Cosine Similarity`)

نحسب زاوية التشابه بين كل جملة $S_i$ والجملة التي تليها مباشرة $S_{i+1}$:
$$\text{Sim}(E_i, E_{i+1}) = \frac{E_i \cdot E_{i+1}}{\|E_i\| \|E_{i+1}\|}$$

In [5]:
print("🔍 درجات التشابه الدلالي بين كل جملة والجملة التالية لها:\n")

similarity_scores = []
for i in range(len(sentences) - 1):
    sim = cosine_similarity([embeddings[i]], [embeddings[i + 1]])[0][0]
    similarity_scores.append(sim)
    print(f"الترابط بين الجملة [{i+1}] والجملة [{i+2}] = {sim:.4f}")
    print(f"   الجملة {i+1}: '{sentences[i][:45]}...'")
    print(f"   الجملة {i+2}: '{sentences[i+1][:45]}...'")
    if sim < 0.2:
        print("   🚨 هبوط دلالي حاد (Cliff Drop)! نقطة انفصال واضحة بين الموضوعين.\n")
    else:
        print("   🔗 ترابط دلالي مستمر.\n")

🔍 درجات التشابه الدلالي بين كل جملة والجملة التالية لها:

الترابط بين الجملة [1] والجملة [2] = 0.8248
   الجملة 1: 'LangChain is a framework for building applica...'
   الجملة 2: 'LangChain provides modular abstraction to com...'
   🔗 ترابط دلالي مستمر.

الترابط بين الجملة [2] والجملة [3] = 0.4487
   الجملة 2: 'LangChain provides modular abstraction to com...'
   الجملة 3: 'You can create chains, agents, memory, and re...'
   🔗 ترابط دلالي مستمر.

الترابط بين الجملة [3] والجملة [4] = 0.0225
   الجملة 3: 'You can create chains, agents, memory, and re...'
   الجملة 4: 'The Eiffel Tower is located in Paris....'
   🚨 هبوط دلالي حاد (Cliff Drop)! نقطة انفصال واضحة بين الموضوعين.

الترابط بين الجملة [4] والجملة [5] = 0.3776
   الجملة 4: 'The Eiffel Tower is located in Paris....'
   الجملة 5: 'France is a popular tourist destination....'
   🔗 ترابط دلالي مستمر.



---
## الخطوة 5: حلقة التجميع الدلالي بالعتبة (`Semantic Grouping Loop`)

نقوم بتنفيذ الخوارزمية المشروحة في المحاضرة:
1. نبدأ القطعة الأولى بالجملة الأولى: `current_chunk = [sentences[0]]`.
2. نمر على الجمل من `i = 1` إلى نهاية القائمة.
3. إذا كان التشابه `sim >= threshold`: ندمج الجملة في القطعة الحالية.
4. إذا كان التشابه أقل من العتبة: نحفظ القطعة السابقة، ونبدأ قطعة جديدة بالجملة الحالية.
5. حفظ القطعة الأخيرة عند نهاية الحلقة.

In [6]:
def semantic_chunk_text(sentences, embeddings, threshold=0.7):
    """
    تقسيم النص إلى قطع دلالية بناءً على عتبة التشابه.
    """
    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
        if sim >= threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

# تشغيل الخوارزمية بالعتبة المشروحة بالفيديو (threshold = 0.7)
chunks_07 = semantic_chunk_text(sentences, embeddings, threshold=0.7)

print(f"📦 نتائج التقطيع بالعتبة المحكمة (Threshold = 0.7) -> عدد القطع: {len(chunks_07)}\n")
for idx, chunk in enumerate(chunks_07, 1):
    print(f"--- [القطعة {idx}] ---")
    print(f"{chunk}\n")

📦 نتائج التقطيع بالعتبة المحكمة (Threshold = 0.7) -> عدد القطع: 4

--- [القطعة 1] ---
LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.

--- [القطعة 2] ---
You can create chains, agents, memory, and retrievers.

--- [القطعة 3] ---
The Eiffel Tower is located in Paris.

--- [القطعة 4] ---
France is a popular tourist destination.



---
## الخطوة 6: مقارنة أثر العتبة (`Threshold Tuning Experiment`)

ماذا يحدث لو جعلنا العتبة متوازنة (`threshold = 0.35`) لدمج كل سياق الذكاء الاصطناعي معاً، وسياق فرنسا وسياحتها معاً؟

In [7]:
# تشغيل الخوارزمية بالعتبة المتوازنة (threshold = 0.35)
chunks_035 = semantic_chunk_text(sentences, embeddings, threshold=0.35)

print(f"📦 نتائج التقطيع بالعتبة المتوازنة (Threshold = 0.35) -> عدد القطع: {len(chunks_035)}\n")
for idx, chunk in enumerate(chunks_035, 1):
    print(f"--- [القطعة {idx}] ---")
    print(f"{chunk}\n")

📦 نتائج التقطيع بالعتبة المتوازنة (Threshold = 0.35) -> عدد القطع: 2

--- [القطعة 1] ---
LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.

--- [القطعة 2] ---
The Eiffel Tower is located in Paris. France is a popular tourist destination.



---
## الخطوة 7: المقارنة والتحقق البرمجي النهائي (`Validation & Assertion`)

In [8]:
# التحقق من صحة مخرجات الخوارزمية
assert len(chunks_07) == 4, "يجب أن ينتج 4 قطع عند العتبة 0.7"
assert len(chunks_035) == 2, "يجب أن ينتج قطعتان دلاليتان متكاملتان عند العتبة 0.35"
assert "Eiffel Tower" in chunks_035[1], "القطعة الثانية يجب أن تحتوي على معالم فرنسا"
assert "LangChain" in chunks_035[0], "القطعة الأولى يجب أن تحتوي على مفاهيم لانج تشين"

print("✅ تم التحقق من نجاح خوارزمية التقطيع الدلالي واكتمال كافة الاختبارات بنجاح!")

✅ تم التحقق من نجاح خوارزمية التقطيع الدلالي واكتمال كافة الاختبارات بنجاح!
